In [1]:
import glob, os, random, json
import numpy as np, pandas as pd
random.seed(0)

META = glob.glob("/kaggle/input/**/places_final_FROZEN.csv", recursive=True)[0]
meta = pd.read_csv(META).set_index("id")
labs = sorted(glob.glob("/kaggle/input/**/labels.npz", recursive=True))
print("places:", len(labs), "| meta:", META)

def lut(d):
    t = np.full(256, 255, np.uint8)
    for k, v in d.items(): t[k] = v
    return t
WC = lut({10:1, 20:5, 30:2, 40:4, 50:6, 60:7, 70:8, 80:0, 90:3, 95:1, 100:7})
ES = lut({1:0, 2:1, 4:3, 5:4, 7:6, 8:7, 9:8, 11:253})
DW = lut({i: i for i in range(9)})
LUTS = [WC, WC, ES, ES, DW, DW]
CLS = ["water", "trees", "grass", "flooded", "crops", "shrub", "built", "bare", "snow"]

def consensus(ms):
    votes = np.zeros((9,) + ms[0].shape, np.int8)
    for m in ms:
        for c in range(9): votes[c] += (m == c)
        votes[2] += (m == 253); votes[5] += (m == 253)
    mx = votes.max(0)
    best = votes.argmax(0).astype(np.uint8)
    ties = (votes == mx).sum(0) > 1
    return np.where((mx >= 2) & ~ties, best, 255).astype(np.uint8)

os.makedirs("/kaggle/working/cons", exist_ok=True)
cnt = {s: np.zeros((2, 256)) for s in ["train", "val", "test"]}
src_ok = np.zeros(3); src_n = np.zeros(3)
rows = []
for i, p in enumerate(labs):
    d = os.path.dirname(p); pid = int(os.path.basename(d)); run = os.path.basename(os.path.dirname(d))
    y = np.load(p)["y"]
    m = [LUTS[k][y[:, k]] for k in range(6)]
    c20 = consensus([m[0], m[2], m[4]]); c21 = consensus([m[1], m[3], m[5]])
    np.savez_compressed(f"/kaggle/working/cons/{pid:04d}.npz", cons=np.stack([c20, c21], 1))
    split = meta.loc[pid, "split"]
    ok = int(open(os.path.join(d, "DONE")).read().split(",")[0])
    rows.append(dict(id=pid, run=run, split=split, patches=y.shape[0], months_ok=ok, dir=d))
    if split in cnt:
        cnt[split][0] += np.bincount(c20.ravel(), minlength=256)
        cnt[split][1] += np.bincount(c21.ravel(), minlength=256)
    v = c21 != 255
    for j, k in enumerate([1, 3, 5]):
        good = (m[k] == c21) | ((m[k] == 253) & ((c21 == 2) | (c21 == 5)))
        src_ok[j] += (good & v).sum(); src_n[j] += v.sum()
    if i % 200 == 0: print("done", i, flush=True)

df = pd.DataFrame(rows)
df.drop(columns="dir").to_csv("/kaggle/working/v2_places.csv", index=False)
print("\nplaces per split:", df.split.value_counts().to_dict())
print("patch positions per split:", df.groupby("split").patches.sum().to_dict())

print("\n=== consensus class share (%) of known pixels, 2021 ===")
tab = {}
for s in cnt:
    c = cnt[s][1]; kn = c[:9].sum()
    tab[s] = [100 * c[q] / kn for q in range(9)] + [100 * c[255] / c.sum()]
print(pd.DataFrame(tab, index=CLS + ["ignored % of all"]).round(1).to_string())
print("ignored % 2020 per split:", {s: round(100 * cnt[s][0][255] / cnt[s][0].sum(), 1) for s in cnt})

print("\n=== each 2021 source vs consensus (pixel match %) ===")
for n_, a, b in zip(["worldcover", "esri", "dynamic world"], src_ok, src_n): print(f"{n_:14s} {100 * a / b:.1f}")

# normalization stats from 120 random train places, up to 4 months each
tr = df[(df.split == "train") & (df.months_ok > 0)]
pick = tr.sample(120, random_state=0)
s1 = np.zeros(2); s2 = np.zeros(2); n = 0
for d in pick.dir:
    fs = sorted(glob.glob(os.path.join(d, "20??-??.npz")))
    for f in random.sample(fs, min(4, len(fs))):
        x = np.clip(np.nan_to_num(np.load(f)["x"].astype(np.float32), nan=-35, neginf=-35, posinf=5), -35, 5)
        s1 += x.sum((0, 2, 3)); s2 += (x ** 2).sum((0, 2, 3)); n += x.shape[0] * x.shape[2] * x.shape[3]
mean = s1 / n; std = np.sqrt(s2 / n - mean ** 2)
print("\nS1 normalization (clip -35..5 dB): mean VV,VH =", mean.round(2), "| std =", std.round(2))
json.dump(dict(mean=mean.tolist(), std=std.tolist(), clip=[-35, 5]), open("/kaggle/working/s1_norm.json", "w"))

places: 1124 | meta: /kaggle/input/datasets/vinayybg/lulc-meta/places_final_FROZEN.csv
done 0
done 200
done 400
done 600
done 800
done 1000

places per split: {'train': 894, 'test': 148, 'val': 82}
patch positions per split: {'test': 1556, 'train': 9355, 'val': 820}

=== consensus class share (%) of known pixels, 2021 ===
                  train   val  test
water               5.8   4.5   6.7
trees              38.5  32.9  39.5
grass              15.1  19.7  15.5
flooded             1.5   2.6   1.7
crops              10.2   8.5   8.7
shrub               8.1   8.4   6.2
built               6.8   7.8   5.5
bare               12.4  13.0  13.8
snow                1.5   2.5   2.4
ignored % of all   12.2  14.3  13.0
ignored % 2020 per split: {'train': np.float64(12.2), 'val': np.float64(14.1), 'test': np.float64(12.5)}

=== each 2021 source vs consensus (pixel match %) ===
worldcover     89.7
esri           87.2
dynamic world  78.8

S1 normalization (clip -35..5 dB): mean VV,VH = [-12.61 -20